# Week 3 Homework: Project Dataset Preprocessing and Feature Engineering

Data preprocessing and feature engineering are vital in data mining as they enhance the quality and performance of data mining models. Data preprocessing ensures that raw data is clean, consistent, and ready for analysis, while feature engineering improves the model’s ability to capture patterns and make accurate predictions.

This week, your task is to preprocess and engineer features of the FinMark Corporation datasets.

To carry out this activity, follow the instructions below:

1. Review the company business case. Review the problem statement and download the required datasets.


   1.1 Problem Statement: FinMark Corporation offers financial products like savings accounts, credit cards, and loans, but all customers receive the same options, which may not fit their individual needs. FinMark wants to improve by offering personalized products to different customer groups based on their behavior and preferences.
    
   1.2 Project Deliverables: As their data analyst, your task is to create a data model using clustering techniques to group customers into segments. These segments will help recommend personalized financial products, improving customer satisfaction and relevance, For the next 12 weeks, the company wants you to develop the following:
    
        1.2.1 Milestone 1: Project Data Preparation and Exploratory Analysis - an exploratory data analysis report of the cleaned company datasets.
        
        1.2.2 Milestone 2: Project Clustering and Segmentation - application of clustering techniques to the dataset and identification of initial customer segments.
        
        1.2.3 Terminal Assessment: Project Final Report and Product Recommendations - presentation of finalized analysis and proposed personalized financial products. 
    
   1.3 Datasets: The following are the datasets that you will use for the development. Click on the item you want to download.
   
        1.3.1 Customer Feedback
   
        1.3.2 Product Offering
   
        1.3.3 Transactions

3. Load the datasets to Jupyter Notebook and perform the necessary data processing using Python.
4. Once done, engineer new features to help the model arrive at better solutions to the outline problem statement.
5. Save your output and prepare for a discussion with your team.

In [179]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import scipy as sp
from scipy import stats
import scipy.cluster.hierarchy as sch
from itertools import combinations  # added for phase 2 deduplication
import regex
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

In [180]:
# Set pandas display options to ensure all columns are displayed in a single line
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Set a large width to avoid line breaks

In [181]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/FinMark Corp Raw Datasets/Customer_Feedback_Data.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/FinMark Corp Raw Datasets/Product_Offering_Data.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/FinMark Corp Raw Datasets/Transaction_Data.csv')

# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_feedback_data','product_offering_data','transaction_data']
dataset_titles = ['Customer Feedback','Product Offering','Transactions']

# Preprocessing: Identifying shape of the dataset and data types of each columns

In [183]:
print("*************************************************************************")
print("Homework: Project Dataset Preprocessing and Feature Engineering") #replace
print("*************************************************************************")

# Check for number of columns and rows for each dataset
print("\nPrior to preprocessing:")
for df, dataset_name in zip(datasets, dataset_names):
    print(f"\n{dataset_name} has {df.shape[1]} columns and {df.shape[0]} rows.")
    # Check for data types
    print(f"Data types for each column:\n{df.dtypes}")

*************************************************************************
Homework: Project Dataset Preprocessing and Feature Engineering
*************************************************************************

Prior to preprocessing:

customer_feedback_data has 4 columns and 5050 rows.
Data types for each column:
Customer_ID                  int64
Satisfaction_Score         float64
Feedback_Comments           object
Likelihood_to_Recommend      int64
dtype: object

product_offering_data has 6 columns and 15 rows.
Data types for each column:
Product_ID               int64
Product_Name            object
Product_Type            object
Risk_Level              object
Target_Age_Group       float64
Target_Income_Group     object
dtype: object

transaction_data has 5 columns and 5050 rows.
Data types for each column:
Transaction_ID          int64
Customer_ID             int64
Transaction_Date       object
Transaction_Amount    float64
Transaction_Type       object
dtype: object


In [184]:
for df, dataset_title in zip(datasets, dataset_titles):
    print(f"\n {dataset_title} statistics for numerical categories: \n{df.describe()}")


 Customer Feedback statistics for numerical categories: 
       Customer_ID  Satisfaction_Score  Likelihood_to_Recommend
count  5050.000000         4949.000000              5050.000000
mean    501.949703            5.684785                 5.571287
std     288.806331            3.606511                 2.872577
min       1.000000            1.000000                 1.000000
25%     253.000000            3.000000                 3.000000
50%     502.000000            6.000000                 6.000000
75%     751.750000            8.000000                 8.000000
max    1000.000000           60.000000                10.000000

 Product Offering statistics for numerical categories: 
       Product_ID  Target_Age_Group
count   15.000000               0.0
mean     5.133333               NaN
std      3.159265               NaN
min      1.000000               NaN
25%      2.500000               NaN
50%      5.000000               NaN
75%      7.500000               NaN
max     10.000000    

# Identifying columns with null values

In [186]:
# Function to check for missing values and print only columns with missing values
def check_missing_values(datasets, dataset_titles):
    """
    Checks for missing values in the datasets and prints the variables with missing values.

    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to check.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    """
    for df, dataset_title in zip(datasets, dataset_titles):
        # print(f"\nMissing values for {dataset_name}:")
        missing_values = df.isnull().sum()  # Count missing values in each column
        missing_values = missing_values[missing_values > 0]  # Filter only those with missing values
        
        if not missing_values.empty:
            print(f"\nVariables with missing values in {dataset_title}:")
            for column, count in missing_values.items():
                print(f"{column}: {count} missing values")
        else:
            print(f"\nNo missing values found in {dataset_title}.")
            
# Call the function to check for and print columns with missing values
check_missing_values(datasets, dataset_titles)


Variables with missing values in Customer Feedback:
Satisfaction_Score: 101 missing values

Variables with missing values in Product Offering:
Target_Age_Group: 15 missing values

Variables with missing values in Transactions:
Transaction_Amount: 100 missing values


# Calculating missingness rate per column

In [188]:
# Calculate missingness for each column
def missingness_rate(datasets, dataset_titles):
    for df, dataset_title in zip(datasets, dataset_titles):
        missingness = df.isnull().sum() / len(df) * 100
        # Display missingness percentage for each column
        print(f"\nMissingness rate per column in {dataset_title}:\n{missingness}")

missingness_rate(datasets, dataset_titles)


Missingness rate per column in Customer Feedback:
Customer_ID                0.0
Satisfaction_Score         2.0
Feedback_Comments          0.0
Likelihood_to_Recommend    0.0
dtype: float64

Missingness rate per column in Product Offering:
Product_ID               0.0
Product_Name             0.0
Product_Type             0.0
Risk_Level               0.0
Target_Age_Group       100.0
Target_Income_Group      0.0
dtype: float64

Missingness rate per column in Transactions:
Transaction_ID        0.000000
Customer_ID           0.000000
Transaction_Date      0.000000
Transaction_Amount    1.980198
Transaction_Type      0.000000
dtype: float64


# Display unique values per column

In [190]:
# Function to display unique values per column for a dataset
def display_unique_values(datasets, dataset_names):
    """
    Displays the unique values in each column of the given datasets.
    
    Parameters:
    datasets (list of pd.DataFrame): List of datasets to inspect.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    
    Returns:
    None
    """
    for df, dataset_name in zip(datasets, dataset_names):
        print('**********************************************************')
        print(f"--- Unique Values in Columns for {dataset_name} ---")
        print('**********************************************************')
        for column in df.columns:
            unique_values = df[column].unique()
            print(f"Column: {column}")
            print(f"Unique Values ({len(unique_values)}): {unique_values}")
            print(' ' * 50)

# Call the function to display unique values
display_unique_values(datasets, dataset_names)

**********************************************************
--- Unique Values in Columns for customer_feedback_data ---
**********************************************************
Column: Customer_ID
Unique Values (1000): [   1    2    3    4    5    6    7    8    9   10   11   12   13   14
   15   16   17   18   19   20   21   22   23   24   25   26   27   28
   29   30   31   32   33   34   35   36   37   38   39   40   41   42
   43   44   45   46   47   48   49   50   51   52   53   54   55   56
   57   58   59   60   61   62   63   64   65   66   67   68   69   70
   71   72   73   74   75   76   77   78   79   80   81   82   83   84
   85   86   87   88   89   90   91   92   93   94   95   96   97   98
   99  100  101  102  103  104  105  106  107  108  109  110  111  112
  113  114  115  116  117  118  119  120  121  122  123  124  125  126
  127  128  129  130  131  132  133  134  135  136  137  138  139  140
  141  142  143  144  145  146  147  148  149  150  151  152  153  154

# Check for duplicate rows and remove them

In [192]:
def deduplication_phase1(datasets, dataset_names, dataset_titles):
    """
    Performs the first phase of deduplication on multiple datasets.
    
    Parameters:
    datasets (list of pd.DataFrame): List of dataframes to deduplicate.
    dataset_names (list of str): List of dataset names corresponding to the dataframes.
    dataset_titles (list of str): List of dataset titles for printing in the output.
    
    Returns:
    dict: A dictionary where the keys are cleaned dataset names and the values are the cleaned dataframes.
    """
    deduplicated_datasets_phase1 = {}

    for df, dataset_name, dataset_title in zip(datasets, dataset_names, dataset_titles):
        
        # Check for duplicates
        duplicates_phase1 = df.duplicated()  # This returns a boolean Series    
        
        # Count the number of duplicate rows
        num_duplicates_phase1 = duplicates_phase1.sum()  # Sum of True values gives the count
        
        # Print the number of duplicate rows
        if num_duplicates_phase1 > 0:
            print("\n------------------------------------------------------------------")
            print(f"--- First Phase of Deduplication for {dataset_title} ---")
            print("------------------------------------------------------------------")
            print(f"Findings: {dataset_title} has {num_duplicates_phase1} duplicate rows.")
        else:
            print(f"Findings: {dataset_title} has no duplicate rows.")
        
        # Remove duplicates and keep the first occurrence
        df_deduplication_phase1 = df.drop_duplicates(keep='first')
        
        # Print the number of duplicates removed
        num_removed_phase1 = len(df) - len(df_deduplication_phase1)
        print(f"Action/s: Removed {num_removed_phase1} duplicate rows from {dataset_title}.")
        
        # Print the number of rows after removal
        print(f"Results: {dataset_title} now has {len(df_deduplication_phase1)} rows after the first phase of deduplication.")
        
        # Save cleaned dataset with modified name
        deduplicated_dataset_name = dataset_name + "_deduplication_phase1"
        deduplicated_datasets_phase1[deduplicated_dataset_name] = df_deduplication_phase1
        
        # Save cleaned dataset to CSV
        df_deduplication_phase1.to_csv(f"{deduplicated_dataset_name}.csv", index=False)
        print(f"Cleaned dataset saved as {deduplicated_dataset_name}.csv")

    return deduplicated_datasets_phase1

# Call deduplication_phase1 for all datasets
deduplicated_datasets_phase1 = deduplication_phase1(datasets, dataset_names, dataset_titles)


------------------------------------------------------------------
--- First Phase of Deduplication for Customer Feedback ---
------------------------------------------------------------------
Findings: Customer Feedback has 81 duplicate rows.
Action/s: Removed 81 duplicate rows from Customer Feedback.
Results: Customer Feedback now has 4969 rows after the first phase of deduplication.
Cleaned dataset saved as customer_feedback_data_deduplication_phase1.csv

------------------------------------------------------------------
--- First Phase of Deduplication for Product Offering ---
------------------------------------------------------------------
Findings: Product Offering has 5 duplicate rows.
Action/s: Removed 5 duplicate rows from Product Offering.
Results: Product Offering now has 10 rows after the first phase of deduplication.
Cleaned dataset saved as product_offering_data_deduplication_phase1.csv

------------------------------------------------------------------
--- First Phase

# Identifying duplicates of unique identifiers

In [194]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/customer_feedback_data_deduplication_phase1.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/product_offering_data_deduplication_phase1.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/transaction_data_deduplication_phase1.csv')


# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_feedback_data','product_offering_data','transaction_data']
dataset_titles = ['Customer Feedback','Product Offering','Transactions']
unique_id_columns = ['Customer_ID', 'Product_ID', 'Transaction_ID']

# Phase 2 of Deduplication: Count Duplicates in Unique ID and Print Unique Identifiers
def find_duplicate_uid(datasets, dataset_names, dataset_titles, unique_id_columns):
    for df, dataset_name, dataset_title, unique_id_column in zip(datasets, dataset_names, dataset_titles, unique_id_columns):
        print("\n---------------------------------------------------------------")
        print(f"--- Second Phase of Deduplication for {dataset_title} ---")
        print("---------------------------------------------------------------")
        
        # Check for duplicates based on unique identifier
        duplicates_unique_id = df.duplicated(subset=[unique_id_column], keep=False)  
        
        # Count the number of duplicate rows
        num_duplicates_phase2 = duplicates_unique_id.sum()  # Sum of True values gives the count
        
        # Print the number of duplicate rows
        if num_duplicates_phase2 > 0:
            print(f"Findings: {dataset_title} has {num_duplicates_phase2} duplicate rows based on '{unique_id_column}'.")
            
            # Print the unique identifiers of the duplicates
            duplicate_ids = df[duplicates_unique_id][unique_id_column].unique()  # Get unique identifiers
            print(f"Duplicate Unique Identifiers: \n{duplicate_ids}")
        else:
            print(f"Findings: {dataset_title} has no duplicate rows based on '{unique_id_column}'.")

# Run deduplication for all datasets
find_duplicate_uid(datasets, dataset_names, dataset_titles, unique_id_columns)


---------------------------------------------------------------
--- Second Phase of Deduplication for Customer Feedback ---
---------------------------------------------------------------
Findings: Customer Feedback has 4952 duplicate rows based on 'Customer_ID'.
Duplicate Unique Identifiers: 
[   1    2    3    4    5    6    7    8    9   10   11   12   13   14
   15   16   17   18   19   20   21   22   23   24   25   26   27   28
   29   30   31   32   33   34   35   36   37   38   39   40   42   43
   44   46   47   48   49   50   51   52   53   54   55   56   57   58
   59   61   62   63   64   65   66   67   69   70   71   72   73   74
   75   76   77   78   79   80   81   82   83   84   85   86   87   88
   89   90   91   92   93   94   96   97   98   99  100  101  102  103
  104  105  106  107  108  109  110  111  112  113  114  115  116  117
  118  119  120  121  122  123  124  125  126  127  128  129  130  131
  132  133  134  135  136  137  138  139  140  141  142  143  144

In [208]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/customer_feedback_data_deduplication_phase1.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/transaction_data_deduplication_phase1.csv')

# Define datasets and dataset_names
datasets = [data1, data3] #replace
dataset_names = ['customer_feedback_data','transaction_data']
dataset_titles = ['Customer Feedback','Transactions']

# Function to delete records with 1 or more missing values and save the cleaned datasets
def delete_missing_records(datasets, dataset_names, columns_to_search):
    for dataset, dataset_name in zip(datasets, dataset_names):
        print(f"\nProcessing dataset: {dataset_name}")
        
        # Select columns to search for missing values
        columns = columns_to_search[dataset_name]
        
        # Count the number of missing values across the specified columns in each record
        dataset['missing_count'] = dataset[columns].isnull().sum(axis=1)
        
        # Filter out records with 1 or more missing values
        cleaned_dataset = dataset[dataset['missing_count'] < 1].drop(columns=['missing_count'])
        
        # Save the cleaned dataset to a CSV file with "_deleted_missing_records" appended to the filename
        cleaned_filename = f'/Users/fritzcabalhin/Desktop/3. Data Mining Principles/{dataset_name}_deleted_missing_records.csv'
        cleaned_dataset.to_csv(cleaned_filename, index=False)
        
        print(f"Saved cleaned dataset as: {cleaned_filename}")
        print(f"Number of records removed: {len(dataset) - len(cleaned_dataset)}")
        print(f"Number of remaining records: {len(cleaned_dataset)}\n")

# Call the function with the datasets and corresponding names
delete_missing_records(datasets, dataset_names, columns_to_search)


Processing dataset: customer_feedback_data
Saved cleaned dataset as: /Users/fritzcabalhin/Desktop/3. Data Mining Principles/customer_feedback_data_deleted_missing_records.csv
Number of records removed: 100
Number of remaining records: 4869


Processing dataset: transaction_data
Saved cleaned dataset as: /Users/fritzcabalhin/Desktop/3. Data Mining Principles/transaction_data_deleted_missing_records.csv
Number of records removed: 100
Number of remaining records: 4900



In [210]:
# Load the datasets
data1 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/customer_feedback_data_deleted_missing_records.csv')
data2 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/product_offering_data_deduplication_phase1.csv')
data3 = pd.read_csv('/Users/fritzcabalhin/Desktop/3. Data Mining Principles/transaction_data_deleted_missing_records.csv')


# Define datasets and dataset_names
datasets = [data1, data2, data3] #replace
dataset_names = ['customer_feedback_data','product_offering_data','transaction_data']
dataset_titles = ['Customer Feedback','Product Offering','Transactions']

# Calculate missingness for each column
def missingness_rate(datasets, dataset_titles):
    for df, dataset_title in zip(datasets, dataset_titles):
        missingness = df.isnull().sum() / len(df) * 100
        # Display missingness percentage for each column
        print(f"\nMissingness rate per column in {dataset_title}:\n{missingness}")

missingness_rate(datasets, dataset_titles)


Missingness rate per column in Customer Feedback:
Customer_ID                0.0
Satisfaction_Score         0.0
Feedback_Comments          0.0
Likelihood_to_Recommend    0.0
dtype: float64

Missingness rate per column in Product Offering:
Product_ID               0.0
Product_Name             0.0
Product_Type             0.0
Risk_Level               0.0
Target_Age_Group       100.0
Target_Income_Group      0.0
dtype: float64

Missingness rate per column in Transactions:
Transaction_ID        0.0
Customer_ID           0.0
Transaction_Date      0.0
Transaction_Amount    0.0
Transaction_Type      0.0
dtype: float64


# Feature Engineering Ideas

1. Customer_Feedback_Data.csv

        1.1 Feedback_ID: Add a new column to insert unique ID for each unique feedback in the Customer Feedback Data. Please do this for me via iteration using Python.

        1.2 Feedback_Date: Add a new column that will automatically add the date when feedback was received from the customer.
       
        1.3 Normalized Scores: Normalize Satisfaction_Score and Likelihood_to_Recommend to a 0-1 range for comparability. - How do you propose for this to be done? Please give examples and a Python code.
       
        1.4 Sentiment Analysis: Perform sentiment scoring of the Feedback_Comments column (e.g., Excellent = 5, Unsatisfactory = 1) to create a numerical sentiment feature.
       
        1.5 Customer Loyalty Index: Combine Satisfaction_Score, Likelihood_to_Recommend, and sentiment scores into a single index. - How do you propose for this to be done? Please give examples and a Python code.


3. Product_Offering_Data.csv

        2.1 Impute Missing Age and Income Groups: Use clustering or logical rules (based on Risk_Level and Product_Type) to fill in the Target_Age_Group and Target_Income_Group.
       
        2.2 Diversity of Offerings: Create a feature summarizing the number of product types offered to each target group (e.g., count the unique Product_Type).


4. Transaction_Data.csv

        3.1 Transaction Frequency: Calculate the number of transactions per customer within specific timeframes (monthly, quarterly, etc.).
       
        3.2 Average Transaction Amount: Aggregate and average Transaction_Amount per customer.
       
        3.3 Spending Habits: Break down Transaction_Amount into categories (e.g., Purchase vs. Loan Payment) to see where each customer spends.
       
        3.4 Recency, Frequency, Monetary Value (RFM):
    
           3.4.1 Recency: Days since last transaction for each customer.
    
           3.4.2 Frequency: Total number of transactions.
    
           3.4.3 Monetary: Sum of all transaction amounts.
    
       3.5 Transaction Seasonality: Extract month or quarter from Transaction_Date to understand seasonal spending patterns.


5. Derived Features Across Datasets

        4.1 Cross-Dataset Features:
    
           4.1.1 Merge Transaction_Data and Customer_Feedback_Data to link spending behavior with satisfaction.
    
           4.1.2 Merge Transaction_Data and Product_Offering_Data to understand the popularity of specific products among different customer groups.
    
       4.2 Spending vs. Satisfaction: Analyze if higher spending correlates with higher satisfaction or likelihood to recommend.